In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, re, sys, json, subprocess, tempfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
TOKEN = s.get_secret("GITHUB_TOKEN")
os.environ["KAGGLE_USERNAME"] = s.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = s.get_secret("KAGGLE_KEY")
URL = f"https://{TOKEN}@github.com/jonsnow-org/Ttbik.git"
subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=True)

code = Path("/kaggle/working/code")
if not code.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch",
                    "claude/free-services-marketplace-h6rwk2", URL, str(code)], check=True)
sys.path.insert(0, str(code / "ai-system/scripts"))
from kaggle_auto_resume import _load_kaggle_api
from sham_registry import RateLimitedApi, list_all_kernels, fetch_failure_log, _get

SECRET_PATTERNS = [r"gh[pousr]_[A-Za-z0-9]{20,}", r"github_pat_[A-Za-z0-9_]{20,}", r"hf_[A-Za-z0-9]{20,}",
                   r"\d{8,10}:AA[A-Za-z0-9_-]{30,}", r"sk-[A-Za-z0-9_-]{20,}", r"eyJ[A-Za-z0-9_-]{20,}\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+"]
def scrub(t):
    for p in SECRET_PATTERNS:
        t = re.sub(p, "***مفتاح_محذوف***", t)
    return t

api = RateLimitedApi(_load_kaggle_api())
out = Path(tempfile.mkdtemp()) / "r"
subprocess.run(["git", "init", "-q", "-b", "sham-notebooks-review", str(out)], check=True)
(out / "vercel.json").write_text('{"git":{"deploymentEnabled":false}}')
index = []
for k in list_all_kernels(api):
    ref = _get(k, "ref"); slug = ref.split("/")[-1]
    d = out / slug; d.mkdir(parents=True, exist_ok=True)
    try:
        api.kernels_pull(ref, str(d), metadata=True)
    except Exception as e:
        (d / "PULL_ERROR.txt").write_text(str(e))
    try:
        st = api.kernels_status(ref)
        status = str(_get(st, "status", default="?")); fail = str(_get(st, "failure_message", default="") or "")
    except Exception as e:
        status, fail = "?", str(e)
    if "error" in status.lower() or "fail" in status.lower():
        try:
            (d / "FAILURE_LOG.txt").write_text(fetch_failure_log(api, ref, Path(tempfile.mkdtemp()))[-20000:])
        except Exception as e:
            (d / "FAILURE_LOG.txt").write_text(f"تعذّر جلب السجل: {e}")
    for f in d.rglob("*"):
        if f.is_file() and f.suffix in (".ipynb", ".py", ".json", ".txt"):
            f.write_text(scrub(f.read_text(encoding="utf-8", errors="ignore")), encoding="utf-8")
    index.append({"ref": ref, "title": _get(k, "title"), "status": status, "failure": scrub(fail)})
    print(f"✔ {ref} — {status}")

(out / "INDEX.json").write_text(json.dumps(index, ensure_ascii=False, indent=2), encoding="utf-8")
g = ["git", "-c", "user.name=sham-review", "-c", "user.email=sham@ttbik.local"]
subprocess.run(g + ["add", "-A"], cwd=out, check=True)
subprocess.run(g + ["commit", "-q", "-m", "notebooks for review"], cwd=out, check=True)
subprocess.run(["git", "push", "-q", "-f", URL, "sham-notebooks-review"], cwd=out, check=True)
print(f"\nتم رفع {len(index)} دفتراً للمراجعة. أخبري Claude.")